# 01 · Baseline evaluation — Qwen3-ASR-1.7B on Vietnamese

Measures the **out-of-the-box** WER/CER of `Qwen/Qwen3-ASR-1.7B-hf` on the
Vietnamese test split, overall and per duration bucket (5–30 s / 30–60 s).

**Hardware:** RTX 5080 (16 GB) — inference only, ~4 GB VRAM.

**Notes baked in from setup:**
- Audio is passed to the processor as a **numpy array** (not a file path): the
  processor's file loader uses torchcodec+FFmpeg, and FFmpeg 4 on this machine
  is buggy. Passing arrays avoids it.
- Requires `transformers>=5.14` (adds the `qwen3_asr` architecture).

Run top-to-bottom (Kernel → Restart & Run All). Writes results to `results/`.

In [ ]:
import os, random, numpy as np, torch
from dotenv import load_dotenv
load_dotenv()
assert os.environ.get("HF_TOKEN"), "Set HF_TOKEN in .env (see .env.example)"

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("torch", torch.__version__, "| device:", DEVICE,
      "|", torch.cuda.get_device_name(0) if DEVICE == "cuda" else "")

torch 2.8.0+cu128 | device: cuda | NVIDIA GeForce RTX 5080


## 1 · Build / load the dataset
Cache-aware: builds `data/vi_asr/` once (streams ~8 h of viVoice), then reuses it. Notebook 2 shares the same cache.

In [2]:
import data_prep
data_prep.prepare_dataset(target_hours=8.0)   # builds once; no-op if cached
splits = data_prep.load_splits()
test = splits["test"]
print({k: len(v) for k, v in splits.items()})
from collections import Counter
print("test buckets:", dict(Counter(test["bucket"])))

[data_prep] cache hit at data/vi_asr; skipping build.
{'train': 980, 'val': 208, 'test': 114}
test buckets: {'5-30': 94, '30-60': 20}


## 2 · Load the model + processor

In [3]:
from transformers import AutoProcessor, AutoModelForMultimodalLM
MODEL_ID = "Qwen/Qwen3-ASR-1.7B-hf"
processor = AutoProcessor.from_pretrained(MODEL_ID)
model = AutoModelForMultimodalLM.from_pretrained(
    MODEL_ID, dtype=torch.bfloat16, attn_implementation="sdpa", device_map=DEVICE,
).eval()
print("loaded", type(model).__name__, model.dtype, model.device)

Loading weights:   0%|          | 0/707 [00:00<?, ?it/s]

loaded Qwen3ASRForConditionalGeneration torch.bfloat16 cuda:0


## 3 · Vietnamese text normalization + WER/CER
Lowercase, NFC-normalize, strip punctuation (keeping Vietnamese diacritics), collapse whitespace — then compute WER and CER with `jiwer`.

The references write numbers as digits (`334%`) while the model transcribes what was spoken (`ba trăm ba mươi bốn phần trăm`), which costs a substitution plus a run of insertions and can push a short utterance to 100% WER. So `normalize_vi` also expands digits to their spoken form on **both** sides and folds the alternative readings onto one variant. Both metrics are reported: `wer` (number-normalized) and `wer_legacy` (without it).

Shared with `show_results.py` via `vi_norm.py` so there is one definition.

In [4]:
from vi_norm import normalize_vi, wer_cer

# normalize_vi expands digits to their spoken form ("334%" -> "ba trăm ba mươi
# bốn phần trăm") on both sides and folds alternative readings (tư/bốn,
# mốt/một, lăm/năm, ngàn/nghìn, linh/lẻ) onto one variant. wer_cer also reports
# wer_legacy/cer_legacy — the same metric without number expansion.

print(normalize_vi("Xin chào, Việt Nam!  "))
print(normalize_vi("Đã đóng sổ với tổng cộng 334%."))
print(wer_cer(["Trong 40 năm tiếp theo."], ["trong bốn mươi năm tiếp theo"]))


xin chào việt nam
đã đóng sổ với tổng cộng ba trăm ba mươi bốn phần trăm
{'wer': 0.0, 'cer': 0.0, 'n': 1, 'wer_legacy': 0.4, 'cer_legacy': 0.36363636363636365}


## 4 · Transcription helper
Passes the raw 16 kHz numpy array straight to the processor.

In [5]:
@torch.no_grad()
def transcribe(row, max_new_tokens=440):
    arr, _ = data_prep.read_audio(row["audio_path"])   # 16 kHz mono
    inputs = processor.apply_transcription_request(
        audio=arr, language="Vietnamese",
    ).to(model.device, model.dtype)
    out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    gen = out[:, inputs["input_ids"].shape[1]:]
    return processor.decode(gen, return_format="transcription_only")[0]

# smoke: one sample
print(transcribe(test[0])[:120])

ngao gác gì mà thấy bao hết rồi ai múa Vinh đã vào bên trong sân khấu rồi, khán giả và ekip đã có mặt đầy đủ để sẵn sàng


In [6]:
# Batched transcription (~3x faster on GPU). Left-padding is REQUIRED for correct
# batched generation; clips are sorted by length so each batch pads efficiently.
processor.tokenizer.padding_side = "left"
BATCH_SIZE = 8   # lower to 4 if you hit CUDA OOM (e.g. many 30-60s clips per batch)

@torch.no_grad()
def transcribe_batch(rows, batch_size=BATCH_SIZE, max_new_tokens=440):
    from tqdm.auto import tqdm
    rows = list(rows)
    order = sorted(range(len(rows)), key=lambda i: rows[i]["duration"])  # group similar lengths
    hyps = [None] * len(rows)
    for s in tqdm(range(0, len(order), batch_size), desc="transcribing (batched)"):
        chunk = order[s:s + batch_size]
        arrs = [data_prep.read_audio(rows[i]["audio_path"])[0] for i in chunk]
        inputs = processor.apply_transcription_request(
            audio=arrs, language="Vietnamese").to(model.device, model.dtype)
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
        gen = out[:, inputs["input_ids"].shape[1]:]
        for i, txt in zip(chunk, processor.decode(gen, return_format="transcription_only")):
            hyps[i] = txt
    return hyps

# smoke: batch of 2 should match single-sample output
print(transcribe_batch([test[0], test[1]])[0][:80])

transcribing (batched):   0%|          | 0/1 [00:00<?, ?it/s]

ngao gác gì mà thấy bao hết rồi ai múa Vinh đã vào bên trong sân khấu rồi, khán 


## 5 · Run evaluation over the test split

In [7]:
import pandas as pd

test_rows = list(test)
hyps = transcribe_batch(test_rows)   # batched, ~3x faster
rows = [{
    "audio_file": os.path.basename(r["audio_path"]),   # for double-checking
    "audio_path": r["audio_path"],
    "channel": r["channel"], "bucket": r["bucket"],
    "duration": round(r["duration"], 2),
    "ref": r["text"], "hyp": h,
} for r, h in zip(test_rows, hyps)]
df = pd.DataFrame(rows)

overall = wer_cer(df["ref"], df["hyp"])
by_bucket = {b: wer_cer(g["ref"], g["hyp"]) for b, g in df.groupby("bucket")}
print("OVERALL:", overall)
for b, m in by_bucket.items():
    print(f"  {b}: WER={m['wer']:.3f} CER={m['cer']:.3f} (n={m['n']})")

transcribing (batched):   0%|          | 0/15 [00:00<?, ?it/s]

OVERALL: {'wer': 0.05957656326932546, 'cer': 0.03033991731740928, 'n': 114, 'wer_legacy': 0.07302804115392375, 'cer_legacy': 0.04000094620807115}
  30-60: WER=0.061 CER=0.032 (n=20)
  5-30: WER=0.059 CER=0.030 (n=94)


## 6 · Save results + inspect samples

In [8]:
import json, pathlib
pathlib.Path("results").mkdir(exist_ok=True)
df.to_csv("results/baseline_predictions.csv", index=False)
with open("results/baseline_metrics.json", "w", encoding="utf-8") as f:
    json.dump({"overall": overall, "by_bucket": by_bucket}, f, ensure_ascii=False, indent=2)
print("saved -> results/baseline_metrics.json, results/baseline_predictions.csv")
df[["audio_file", "bucket", "duration", "ref", "hyp"]].head(5)

saved -> results/baseline_metrics.json, results/baseline_predictions.csv


,audio_file,bucket,duration,ref,hyp
0,seg_000011.wav,5-30,8.96,Ngao gác gì mà heo beo hết rồi ai mua. Vinh đã...,ngao gác gì mà thấy bao hết rồi ai múa Vinh đã...
1,seg_000012.wav,5-30,15.49,"Thôi được rồi. Thôi, tôi cũng không có làm khó...",Thôi được rồi. Thôi tôi cũng không có làm khó ...
2,seg_000013.wav,30-60,34.24,Còn đây là tiền lãi mà lãi mày để quá hạn lâu ...,"còn đây là tiền lãi, mà lãi mày để quá hạn lâu..."
3,seg_000014.wav,5-30,22.20,Ngồi xuống có gì từ từ nói. Này hóa chất thì h...,Ngồi xuống có gì từ từ nói. Này hóa chất thì h...
4,seg_000015.wav,5-30,6.53,Phụ mày đi. Cái gì? 50 triệu hả? Nhưng mà tiền...,Phụ mẹ đi. Cái gì? Năm chục triệu hả? Nhưng mà...


## 7 · (Optional) Raw dataset variant — no merge

Builds a **second** dataset from viVoice using the **raw native clips** (no
concatenation) in `data/vi_asr_raw/`, and evaluates the same baseline model on
it. Results are saved separately as `results/baseline_raw_*` so they don't
overwrite the merged-dataset results above.

> Note: this **re-streams ~8 h** of viVoice (a separate cache), so the first run
> of these cells takes another download. Lower `target_hours` below if you just
> want a quick look. Reuses `transcribe` / `wer_cer` defined earlier.

In [9]:
RAW_DIR = "data/vi_asr_raw"
data_prep.prepare_dataset(target_hours=8.0, out_dir=RAW_DIR, merge=False)  # no concatenation
raw = data_prep.load_splits(RAW_DIR)
raw_test = raw["test"]
from collections import Counter
print({k: len(v) for k, v in raw.items()})
print("raw test buckets:", dict(Counter(raw_test["bucket"])))

[data_prep] cache hit at data/vi_asr_raw; skipping build.
{'train': 5222, 'val': 504, 'test': 1250}
raw test buckets: {'0-5': 892, '5-30': 358}


In [ ]:

import pandas as pd, json

raw_rows_in = list(raw_test)
raw_hyps = transcribe_batch(raw_rows_in)   # batched, ~3x faster
raw_rows = [{
    "audio_file": os.path.basename(r["audio_path"]),
    "audio_path": r["audio_path"],
    "channel": r["channel"], "bucket": r["bucket"],
    "duration": round(r["duration"], 2),
    "ref": r["text"], "hyp": h,
} for r, h in zip(raw_rows_in, raw_hyps)]
raw_df = pd.DataFrame(raw_rows)

raw_overall = wer_cer(raw_df["ref"], raw_df["hyp"])
raw_by = {b: wer_cer(g["ref"], g["hyp"]) for b, g in raw_df.groupby("bucket")}
print("RAW OVERALL:", raw_overall)
for b, m in raw_by.items():
    print(f"  {b}: WER={m['wer']:.3f} CER={m['cer']:.3f} (n={m['n']})")

raw_df.to_csv("results/baseline_raw_predictions.csv", index=False)
with open("results/baseline_raw_metrics.json", "w", encoding="utf-8") as f:
    json.dump({"overall": raw_overall, "by_bucket": raw_by}, f, ensure_ascii=False, indent=2)
print("saved -> results/baseline_raw_metrics.json, results/baseline_raw_predictions.csv")
raw_df[["audio_file", "bucket", "duration", "ref", "hyp"]].head(5)

transcribing (batched):   0%|          | 0/157 [00:00<?, ?it/s]

RAW OVERALL: {'wer': 0.03763822664096581, 'cer': 0.01744199796608566, 'n': 1250, 'wer_legacy': 0.05319094316373158, 'cer_legacy': 0.030706551131515377}
  0-5: WER=0.048 CER=0.023 (n=892)
  5-30: WER=0.029 CER=0.013 (n=358)
saved -> results/baseline_raw_metrics.json, results/baseline_raw_predictions.csv


,audio_file,bucket,duration,ref,hyp
0,clip_000005.wav,0-5,1.96,Bản báo cáo miêu tả tâm trạng.,bản báo cáo miêu tả tâm trạng.
1,clip_000038.wav,0-5,2.39,Nhưng vẫn có sự sống trong điều kiện khắc nghi...,nhưng vẫn có sự sống trong điều kiện khắc nghi...
2,clip_000040.wav,0-5,2.99,"Còn hiện tại, tôi nghĩ rằng đó là 100% công việc.","còn hiện tại, tôi nghĩ rằng đó là một trăm phầ..."
3,clip_000047.wav,0-5,1.19,Hãy cùng bắt đầu nhá!,hãy cùng bắt đầu nhá.
4,clip_000050.wav,0-5,3.41,Anh ta cũng làm như vậy ở tầng 2 và lên đến nó...,Anh ta cũng làm như vậy ở tầng hai và lên đến ...


## 8 · External Vietnamese benchmarks (VIVOS, Common Voice, VLSP 2020)

Places the baseline next to the datasets PhoWhisper reports on. **These cells are
independent of sections 5 and 7** — they only need the model/processor from
section 2, so you can run sections 1–3 then jump here.

| benchmark | source | split | n |
|---|---|---|---|
| `vivos` | `htdung167/vivos-preprocessed-v2` | test | 760 |
| `cmv_vi` | `fixie-ai/common_voice_17_0` (`vi`) | test | 1274 |
| `vlsp2020_100h` | `data/vi_mix/heldout_vlsp2020_100h` | held-out slice | ~5% of 11h |

> **VLSP needs the mixture built first.** It has no public test split, and
> notebook 2 now *trains* on that corpus, so the benchmark reads the held-out
> slice assigned by transcript hash in `mixture.prepare_mixture()`. Run that once
> (notebook 2's data cell does it) or these cells will tell you to. Any earlier
> VLSP number sampled from the train stream is superseded — it was measured on
> rows the mixture now trains on.

**Two caveats on comparing to the PhoWhisper table.**

1. PhoWhisper's WER uses its own text normalization, which isn't fully specified.
   The `PhoWhisper-large` column is a **reference point, not a like-for-like
   baseline** — treat a point or two of gap as noise.
2. **VLSP carries no reference number.** PhoWhisper reports the Task-1/Task-2
   *test* sets, which aren't public. What runs here is the VinBigData 100h corpus
   — VLSP 2020 *training* data — whose transcripts are ~96% accurate by the
   publisher's own estimate, putting a floor under any WER measured on it.

Registry and loader live in `bench.py`; pure logic is unit-tested in
`tests/test_bench.py`.

In [ ]:
import json, pathlib
import bench

processor.tokenizer.padding_side = "left"   # required for correct batched generation
BENCH_BATCH_SIZE = 8   # lower to 4 if you hit CUDA OOM

for k, v in bench.BENCHMARKS.items():
    print(f"{k:<15} {v.repo or v.local_dir:<45} [{v.split}]  limit={v.limit or 'full'}")


In [ ]:
BENCH_TO_RUN = ["vivos", "cmv_vi", "vlsp2020_100h"]   # trim for a quick look

bench_results, bench_frames = bench.run_benchmarks(
    model, processor, BENCH_TO_RUN,
    token=os.environ.get("HF_TOKEN"), batch_size=BENCH_BATCH_SIZE,
)


In [ ]:
pathlib.Path("results").mkdir(exist_ok=True)
for name, bdf in bench_frames.items():
    bdf.to_csv(f"results/bench_{name}_predictions.csv", index=False)
with open("results/bench_metrics.json", "w", encoding="utf-8") as f:
    json.dump(bench_results, f, ensure_ascii=False, indent=2)
print("saved -> results/bench_metrics.json + results/bench_<name>_predictions.csv")

table = bench.compare_table(bench_results)
print()
print(table.to_string(index=False))
print("\nPhoWhisper-large numbers are the published reference (different text")
print("normalization); blank = no comparable published number. See the notes above.")
table
